In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

from pathlib import Path

In [2]:
ROOT = Path.cwd().parent

df = pd.read_parquet(ROOT / "data"/"processed"/"ca1_features.parquet")

print("Shape:", df.shape)
print("Products:", df["item_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

Shape: (5832737, 25)
Products: 3049
Date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00


In [3]:
# Define train and validation periods
dates = np.sort(df['date'].unique())

test_dates = dates[-28:]
train_dates = dates[:-28]

train_df = df[df['date'].isin(train_dates)].copy()
test_df = df[df['date'].isin(test_dates)].copy()

print("Train:", train_df["date"].min(), "to", train_df["date"].max())
print("Validation:", test_df["date"].min(), "to", test_df["date"].max())

Train: 2011-01-29 00:00:00 to 2016-03-27 00:00:00
Validation: 2016-03-28 00:00:00 to 2016-04-24 00:00:00


In [4]:
import gc

del df
gc.collect()

0

In [5]:
feature_cols = [
    # Product information
    "item_id", "dept_id", "cat_id",

    # Calendar features
    "day_of_week", "day_of_month", "month", "year", "week_of_year",

    # Historical sales
    "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_28",

    # Retail information
    "snap_CA", "sell_price",
    
    # Event information
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
]

categorical_cols = [
    "item_id", "dept_id", "cat_id",
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
]


numerical_cols = [
    col for col in feature_cols
    if col not in categorical_cols
]

X_train = train_df[feature_cols].copy()
y_train = train_df["sales"]

X_test = test_df[feature_cols].copy()
y_test = test_df["sales"]

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numerical_cols),
        ("cat", 
         make_pipeline(SimpleImputer(strategy="most_frequent"), 
                       OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
         categorical_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training:", X_train_processed.shape)
print("Validation:", X_test_processed.shape)

Training: (5747365, 19)
Validation: (85372, 19)


In [7]:
del X_train, X_test
gc.collect()

114

In [8]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=5,
                                           random_state=42, n_jobs=-1),

    "XGBoost": XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, 
                            objective="reg:squarederror", random_state=42, n_jobs=-1),

    "LightGBM": LGBMRegressor(n_estimators=500, learning_rate=0.05, num_leaves=31, 
                              random_state=42, n_jobs=-1, verbosity=-1),
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train_processed, y_train)

    predictions = np.maximum(model.predict(X_test_processed), 0)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))

    total_actual = y_test.sum()
    wape = (
        np.abs(y_test.to_numpy() - predictions).sum()
        / total_actual * 100
        if total_actual > 0
        else np.nan
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("WAPE (%)")
results_df.reset_index(drop=True)

Training Linear Regression...
Training Random Forest...
Training XGBoost...
Training LightGBM...


,Model,MAE,RMSE,WAPE (%)
0,XGBoost,1.044483,2.042372,69.688239
1,Random Forest,1.045419,2.025338,69.750685
2,LightGBM,1.046891,2.027638,69.848946
3,Linear Regression,1.083248,2.125265,72.274664


In [12]:
results_dir = ROOT / "outputs"
results_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(results_dir / "model_comparison.csv", index=False)

print("Results saved:", results_dir / "model_comparison.csv")

Results saved: /home/ajaya-kumar-patel/Desktop/Placement/Project/retail-demand-forecasting/reports/model_comparison.csv


## recursive forecasting

In [15]:

def recursive_forecast(model, history_df, future_df, preprocessor, feature_cols):
    # Keep actual sales only from before the forecast window
    histories = {
        item_id: group.sort_values("date")["sales"].tolist()
        for item_id, group in history_df.groupby("item_id", sort=False)
    }

    forecast_results = []

    for forecast_date in sorted(future_df["date"].unique()):
        day_df = (
            future_df[future_df["date"] == forecast_date]
            .sort_values("item_id")
            .copy()
        )

        # Recalculate lag and rolling features using available history
        day_df["lag_7"] = [histories[item][-7] for item in day_df["item_id"]]
        day_df["lag_14"] = [histories[item][-14] for item in day_df["item_id"]]
        day_df["lag_28"] = [histories[item][-28] for item in day_df["item_id"]]

        day_df["rolling_mean_7"] = [
            np.mean(histories[item][-7:]) for item in day_df["item_id"]
        ]
        day_df["rolling_mean_28"] = [
            np.mean(histories[item][-28:]) for item in day_df["item_id"]
        ]

        # Predict this date
        X_day = preprocessor.transform(day_df[feature_cols])
        predictions = np.maximum(model.predict(X_day), 0)

        forecast_results.append(pd.DataFrame({
            "date": day_df["date"].to_numpy(),
            "item_id": day_df["item_id"].to_numpy(),
            "actual": day_df["sales"].to_numpy(),
            "prediction": predictions,
        }))

        # Update histories with predictions, never actual test sales
        for item_id, prediction in zip(day_df["item_id"], predictions):
            histories[item_id].append(float(prediction))

    return pd.concat(forecast_results, ignore_index=True)

In [16]:

results = []
forecast_outputs = {}

for name, model in trained_models.items():
    print(f"Recursive forecasting: {name}")

    forecasts = recursive_forecast(
        model=model,
        history_df=train_df,
        future_df=test_df,
        preprocessor=preprocessor,
        feature_cols=feature_cols,
    )

    actual = forecasts["actual"].to_numpy()
    predicted = forecasts["prediction"].to_numpy()

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    total_actual = actual.sum()
    wape = (
        np.abs(actual - predicted).sum() / total_actual * 100
        if total_actual > 0 else np.nan
    )

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
    })

    forecast_outputs[name] = forecasts

recursive_results_df = (
    pd.DataFrame(results)
    .sort_values("WAPE (%)")
    .reset_index(drop=True)
)

display(recursive_results_df)

# Save separately from the original non-recursive comparison
recursive_results_df.to_csv(
    results_dir / "recursive_28day_comparison.csv",
    index=False,
)

for name, forecasts in forecast_outputs.items():
    filename = name.lower().replace(" ", "_")
    forecasts.to_csv(
        results_dir / f"{filename}_recursive_28day_forecasts.csv",
        index=False,
    )

print("Recursive evaluation results saved to:", results_dir)


Recursive forecasting: Linear Regression
Recursive forecasting: Random Forest
Recursive forecasting: XGBoost
Recursive forecasting: LightGBM


,Model,MAE,RMSE,WAPE (%)
0,XGBoost,1.095257,2.180471,73.075886
1,Random Forest,1.120558,2.167142,74.764034
2,LightGBM,1.122399,2.164666,74.886863
3,Linear Regression,1.138114,2.222596,75.935336


Recursive evaluation results saved to: /home/ajaya-kumar-patel/Desktop/Placement/Project/retail-demand-forecasting/reports
